# Homework 1 - Data Mining
**Authors:** Toni Fuentes Bauzà and Carles Westendorf Vidal

We will try a topic discovery strategy, and even though we have labels, we will try to find better clusters based on meaning.

### Loading database

We will load the emotions database as the assignament statement suggests.

In [1]:
from datasets import load_dataset

# Load the emotion dataset from Hugging Face
print("Loading the 'emotion' dataset...")
emotions = load_dataset("emotion")

# Extract the texts and labels from the training subset
texts = emotions["train"]["text"]
labels = emotions["train"]["label"]

# Print a quick summary to verify everything loaded correctly
print(f"Successfully loaded {len(texts)} training texts.")
print(f"First text: '{texts[0]}'")
print(f"First label (integer): {labels[0]}")

Loading the 'emotion' dataset...


Successfully loaded 16000 training texts.
First text: 'i didnt feel humiliated'
First label (integer): 0


### Semantic embedding

We will use the pre-trained `all-MiniLM-L6-v2` model to obtain a vector out of each text entry, so that it captures the meaning of the text.

In [2]:
from sentence_transformers import SentenceTransformer

# Load the model used in chapter 5
print("Loading the Sentence Transformer model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all the texts
print("Generating embeddings (this might take a few moments)...")
embeddings = embedding_model.encode(texts, show_progress_bar=True)

print(f"Embeddings generated successfully. Shape: {embeddings.shape}")

Loading the Sentence Transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings (this might take a few moments)...


Batches:   0%|          | 0/500 [00:00<?, ?it/s]

Embeddings generated successfully. Shape: (16000, 384)


### Dimentionality reduction

We will use **UMAP** (Uniform Manifold Approximation and Projection) to reduce the dimentionality of these vectors from 384 to only 5. This algorithm is able to preserve both local and global structure of the data, so it facilitates the following clustering process.

In [3]:
import umap

# Configure UMAP to reduce dimensions
# n_neighbors controls the balance between local and global structure
# n_components is our target number of dimensions (5 is a standard choice for HDBSCAN)
print("Reducing dimensions with UMAP (this may take a minute)...")
umap_model = umap.UMAP(
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# Apply the reduction to our embeddings
umap_embeddings = umap_model.fit_transform(embeddings)

print(f"UMAP reduction complete. New shape: {umap_embeddings.shape}")

Reducing dimensions with UMAP (this may take a minute)...


C:\Users\usuari\Documents\TopicClusteringDataMining\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP reduction complete. New shape: (16000, 5)


### Clustering algorithm

Now, we will apply the **HDBSCAN** clustering algorithm, which calculates the number of clusters and labels uncategorized texts as `-1`.

In [4]:
import hdbscan

# Configure HDBSCAN to find clusters
# min_cluster_size dictates the smallest grouping we consider a "topic"
# We will start with 50, as in chapter 5
print("Clustering vectors with HDBSCAN...")
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=50,
    metric='euclidean',
    cluster_selection_method='eom'
)

# Predict the clusters based on our UMAP embeddings
clusters = hdbscan_model.fit_predict(umap_embeddings)

# Calculate how many distinct topics were found (excluding the -1 noise cluster)
num_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
print(f"HDBSCAN discovered {num_clusters} distinct semantic topics.")

Clustering vectors with HDBSCAN...
HDBSCAN discovered 54 distinct semantic topics.


### Topic analysis

Once we have our clusters, we need to find its topics. This will be performed by extracting the top TF-IDF keywords for each cluster.

Also, we will assign the original integer labels to its corresponding emotion and generate a crosstab to check the topic discoveries.

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a DataFrame with the texts, original labels, and new clusters
df = pd.DataFrame({
    "Text": texts,
    "Emotion_Label_Int": labels,
    "Topic_Cluster": clusters
})

# Map the integer labels to actual emotion names
emotion_map = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
df['Emotion_Name'] = df['Emotion_Label_Int'].map(emotion_map)

# Define our keyword extraction function (c-TF-IDF approach)
def extract_cluster_keywords(dataframe, cluster_column, top_n=5):
    print(f"Analyzing '{cluster_column}' Keywords:")
    print("=" * 70)
    valid_clusters = dataframe[dataframe[cluster_column] != -1]
    cluster_texts = valid_clusters.groupby(cluster_column)['Text'].apply(' '.join).reset_index()

    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(cluster_texts['Text'])
    feature_names = vectorizer.get_feature_names_out()

    for i, cluster_id in enumerate(cluster_texts[cluster_column]):
        cluster_scores = tfidf_matrix[i].toarray()[0]
        top_indices = cluster_scores.argsort()[-top_n:][::-1]
        top_words = [feature_names[idx] for idx in top_indices]
        cluster_size = len(dataframe[dataframe[cluster_column] == cluster_id])

        print(f"Cluster {cluster_id} (Size: {cluster_size}): {', '.join(top_words).title()}")

extract_cluster_keywords(df, 'Topic_Cluster', top_n=6)

# Generate the final Crosstab!
print("\nAgreement between Discovered Topics and Emotion Labels:")
print("-" * 70)
crosstab_result = pd.crosstab(df['Topic_Cluster'], df['Emotion_Name'])
print(crosstab_result)

Analyzing 'Topic_Cluster' Keywords:
Cluster 0 (Size: 59): Tortured, Feel, Like, Im, Somebody, Feeling
Cluster 1 (Size: 171): Http, Href, Feel, Www, Src, Img
Cluster 2 (Size: 73): Festive, Christmas, Feeling, Feel, Tree, Im
Cluster 3 (Size: 158): Feel, Smart, Intelligent, Stupid, Dumb, Foolish
Cluster 4 (Size: 118): Selfish, Feel, Carefree, Caring, Like, Feeling
Cluster 5 (Size: 65): Weird, Feel, Strange, Defective, Like, Unnatural
Cluster 6 (Size: 50): Insecure, Feel, Confidence, Feeling, Assured, Confident
Cluster 7 (Size: 85): Worthless, Useless, Feel, Useful, Like, Feeling
Cluster 8 (Size: 122): Rich, Generous, Feel, Feeling, Im, Like
Cluster 9 (Size: 87): Accepted, Feel, Welcomed, Belong, Loved, Like
Cluster 10 (Size: 256): Feel, Divine, God, Church, Feeling, Pray
Cluster 11 (Size: 562): Feel, Feeling, Like, Wear, Hair, Wearing
Cluster 12 (Size: 129): Angry, Feel, Pissed, Feeling, Mad, Anger
Cluster 13 (Size: 56): Gloomy, Feel, Feeling, Im, Weather, Bit
Cluster 14 (Size: 144): Agit

### Conclusion and Interpretation

After avaluating the results of our topic discovery strategy we can extract the following meaningful information.

First, there are more than 6,000 unclustered texts which might be telling us that the clustering algorithm was too strict, leaving only 10,000 texts clustered and processed.

Second, the 54 different clusters we obtained do not generally align with the original labels (emotions), in the sense that they are distrubuted along all of the original labels. This is a symptom of the clusters representing different topics of those in the original labels, hence grouping them more precisely under other meanings.

However, we have noticed that some clusters like the 33rd and 44th correspond mainly to one of the orignal emotions (sadness), so there are some topics that align with these emotions. In this case, cluster 33rd's kayword was punishment, and 44th's ashamed and regretful, which match the sadness label.